# A14. 몬티 홀, 생존자 편향, 식중독 문제

**핵심 주제**: 조건부확률의 반직관적 결과

---

## 학습 목표
1. 몬티 홀 문제를 베이즈 정리와 시뮬레이션으로 이해
2. 생존자 편향(Survivorship Bias)의 개념과 실제 사례
3. 조건부확률을 이용한 역학 조사(식중독 원인 분석)

## 1. 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정 (Malgun Gothic)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Seaborn 스타일
sns.set_theme(style='whitegrid', font='Malgun Gothic')

# 고해상도 출력
plt.rcParams['figure.dpi'] = 120

# 재현성을 위한 시드 고정
np.random.seed(42)

print('환경 설정 완료')

---

## Part 1: Monty Hall 문제

### 문제 설명

미국 TV 게임쇼 *"Let's Make a Deal"*에서 유래한 유명한 확률 문제입니다.

**규칙**:
1. 3개의 문 뒤에 각각 **자동차 1대**와 **염소 2마리**가 숨겨져 있다
2. 참가자가 문 하나를 선택한다
3. 진행자(Monty Hall)가 나머지 2개 문 중 **염소가 있는 문 1개**를 연다
4. 참가자에게 **선택을 바꿀 기회**를 준다

**질문**: 선택을 바꾸는 것이 유리한가?

### 1.1 직관적 분석

많은 사람들의 직관:
> "문이 2개 남았으니 확률은 1/2 아닌가? 바꾸나 안 바꾸나 같다!"

**하지만 이 직관은 틀렸습니다!**

### 1.2 수학적 풀이

참가자가 **문 1번**을 선택했다고 가정합니다.

| 자동차 위치 | 진행자가 여는 문 | Stay 결과 | Switch 결과 |
|:-----------:|:----------------:|:---------:|:-----------:|
| 문 1 (1/3) | 문 2 또는 3 | 자동차 | 염소 |
| 문 2 (1/3) | 문 3 | 염소 | 자동차 |
| 문 3 (1/3) | 문 2 | 염소 | 자동차 |

$$P(\text{자동차} \mid \text{Stay}) = \frac{1}{3}$$

$$P(\text{자동차} \mid \text{Switch}) = \frac{2}{3}$$

### 베이즈 정리로 유도

참가자가 문 1을 선택하고, 진행자가 문 3을 열었다면:

$$P(\text{차가 문2} \mid \text{문3 열림}) = \frac{P(\text{문3 열림} \mid \text{차가 문2}) \cdot P(\text{차가 문2})}{P(\text{문3 열림})}$$

- $P(\text{차가 문2}) = \frac{1}{3}$
- $P(\text{문3 열림} \mid \text{차가 문2}) = 1$ (진행자가 반드시 문3을 열어야 함)
- $P(\text{문3 열림}) = P(\text{문3 열림}|\text{차 문1})P(\text{차 문1}) + P(\text{문3 열림}|\text{차 문2})P(\text{차 문2}) + P(\text{문3 열림}|\text{차 문3})P(\text{차 문3})$
- $= \frac{1}{2} \cdot \frac{1}{3} + 1 \cdot \frac{1}{3} + 0 \cdot \frac{1}{3} = \frac{1}{2}$

$$P(\text{차가 문2} \mid \text{문3 열림}) = \frac{1 \times \frac{1}{3}}{\frac{1}{2}} = \frac{2}{3}$$

**결론: Switch가 Stay보다 2배 유리!**

In [ ]:
# === Monty Hall 시뮬레이션 ===

def monty_hall_simulation(n_games=10000):
    """몬티 홀 게임을 n_games번 시뮬레이션한다.
    
    Parameters
    ----------
    n_games : int
        시뮬레이션 횟수
    
    Returns
    -------
    stay_wins : int
        Stay 전략으로 이긴 횟수
    switch_wins : int
        Switch 전략으로 이긴 횟수
    stay_cumulative : np.ndarray
        Stay 전략의 누적 승률
    switch_cumulative : np.ndarray
        Switch 전략의 누적 승률
    """
    doors = [0, 1, 2]  # 문 번호
    stay_results = []
    switch_results = []
    
    for _ in range(n_games):
        # 자동차 위치 (랜덤)
        car = np.random.randint(0, 3)
        # 참가자 선택 (랜덤)
        choice = np.random.randint(0, 3)
        
        # 진행자가 열 수 있는 문: 참가자가 선택하지 않았고, 자동차가 없는 문
        available = [d for d in doors if d != choice and d != car]
        opened = np.random.choice(available)
        
        # Stay 전략: 원래 선택 유지
        stay_win = (choice == car)
        stay_results.append(stay_win)
        
        # Switch 전략: 남은 문으로 변경
        remaining = [d for d in doors if d != choice and d != opened][0]
        switch_win = (remaining == car)
        switch_results.append(switch_win)
    
    stay_results = np.array(stay_results)
    switch_results = np.array(switch_results)
    
    # 누적 승률 계산
    stay_cumulative = np.cumsum(stay_results) / np.arange(1, n_games + 1)
    switch_cumulative = np.cumsum(switch_results) / np.arange(1, n_games + 1)
    
    return (stay_results.sum(), switch_results.sum(),
            stay_cumulative, switch_cumulative)

# 10,000번 시뮬레이션 실행
n_games = 10000
stay_wins, switch_wins, stay_cum, switch_cum = monty_hall_simulation(n_games)

print("=" * 60)
print(f" Monty Hall 시뮬레이션 결과 ({n_games:,}번 게임)")
print("=" * 60)
print()
print(f"  Stay 전략:   {stay_wins:,}승 / {n_games:,}게임 = {stay_wins/n_games:.4f} ({stay_wins/n_games*100:.1f}%)")
print(f"  Switch 전략: {switch_wins:,}승 / {n_games:,}게임 = {switch_wins/n_games:.4f} ({switch_wins/n_games*100:.1f}%)")
print()
print(f"  이론값: Stay = 1/3 ≈ 0.3333, Switch = 2/3 ≈ 0.6667")
print(f"  Switch가 Stay보다 약 {switch_wins/stay_wins:.1f}배 유리!")

In [ ]:
# === 시뮬레이션 결과 시각화 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 왼쪽: 누적 승률 수렴 그래프 ---
ax1 = axes[0]
games = np.arange(1, n_games + 1)

ax1.plot(games, stay_cum, color='#e74c3c', linewidth=1.5, alpha=0.8, label='Stay 전략')
ax1.plot(games, switch_cum, color='#2980b9', linewidth=1.5, alpha=0.8, label='Switch 전략')

ax1.axhline(y=1/3, color='#e74c3c', linestyle='--', alpha=0.5, label='이론값 1/3')
ax1.axhline(y=2/3, color='#2980b9', linestyle='--', alpha=0.5, label='이론값 2/3')

ax1.set_xlabel('게임 횟수', fontsize=11)
ax1.set_ylabel('누적 승률', fontsize=11)
ax1.set_title('시뮬레이션: 누적 승률의 수렴', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

# --- 오른쪽: 최종 승률 비교 바 차트 ---
ax2 = axes[1]
strategies = ['Stay\n(선택 유지)', 'Switch\n(선택 변경)']
win_rates = [stay_wins / n_games, switch_wins / n_games]
theory_rates = [1/3, 2/3]
colors = ['#e74c3c', '#2980b9']

bars = ax2.bar(strategies, win_rates, color=colors, alpha=0.8, width=0.5,
               edgecolor='white', linewidth=2)

# 이론값 표시
for i, (bar, theory) in enumerate(zip(bars, theory_rates)):
    ax2.hlines(y=theory, xmin=bar.get_x(), xmax=bar.get_x() + bar.get_width(),
               colors='black', linestyles='--', linewidth=2)
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f'{win_rates[i]:.1%}', ha='center', fontsize=14, fontweight='bold')
    ax2.text(bar.get_x() + bar.get_width() + 0.05, theory,
             f'이론: {theory:.4f}', fontsize=9, va='center')

ax2.set_ylabel('승률', fontsize=11)
ax2.set_title(f'최종 승률 비교 ({n_games:,}게임)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 0.85)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# === 확률 트리 다이어그램 ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 색상 정의
c_car = '#f39c12'
c_goat = '#95a5a6'
c_win = '#27ae60'
c_lose = '#e74c3c'

# 시작
ax.text(0.5, 5, '참가자:\n문 1 선택', fontsize=10, fontweight='bold',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#ecf0f1', edgecolor='black'))

# 자동차 위치 3가지
positions = [
    (8.5, '문 1에 차\n(1/3)', c_car, '문 2 or 3\n열림', 'Stay: 차!', 'Switch: 염소', c_win, c_lose),
    (5.0, '문 2에 차\n(1/3)', c_car, '문 3 열림\n(염소)', 'Stay: 염소', 'Switch: 차!', c_lose, c_win),
    (1.5, '문 3에 차\n(1/3)', c_car, '문 2 열림\n(염소)', 'Stay: 염소', 'Switch: 차!', c_lose, c_win)
]

for y, car_label, color, opened_label, stay_result, switch_result, stay_color, switch_color in positions:
    # 자동차 위치 가지
    ax.annotate('', xy=(2.5, y), xytext=(1.3, 5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax.text(3, y, car_label, fontsize=9, fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#fef9e7', edgecolor=c_car))
    
    # 진행자가 여는 문
    ax.annotate('', xy=(5, y), xytext=(4, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))
    ax.text(5.5, y, opened_label, fontsize=8, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5'))
    
    # Stay 결과
    ax.annotate('', xy=(7.5, y + 0.5), xytext=(6.5, y + 0.2),
                arrowprops=dict(arrowstyle='->', color=stay_color, lw=1.5))
    ax.text(8.3, y + 0.5, stay_result, fontsize=9, fontweight='bold',
            ha='center', va='center', color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=stay_color))
    
    # Switch 결과
    ax.annotate('', xy=(7.5, y - 0.5), xytext=(6.5, y - 0.2),
                arrowprops=dict(arrowstyle='->', color=switch_color, lw=1.5))
    ax.text(8.3, y - 0.5, switch_result, fontsize=9, fontweight='bold',
            ha='center', va='center', color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=switch_color))

# 결론 박스
ax.text(5, 9.5, 'Stay: 1/3 확률로 승리    |    Switch: 2/3 확률로 승리',
        fontsize=13, fontweight='bold', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#eaf2f8',
                  edgecolor='#2980b9', linewidth=2))

ax.set_title('Monty Hall 문제: 확률 트리', fontsize=14, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

### 1.3 확장: 100개 문 문제

직관을 교정하는 데 가장 효과적인 비유입니다.

In [ ]:
# === 확장: N개의 문으로 일반화 ===

print("=" * 60)
print(" 확장: 문이 N개일 때 Switch 확률")
print("=" * 60)
print()
print("규칙: N개의 문 중 1개 선택 후, 진행자가 N-2개 문을 열고,")
print("      남은 1개 문으로 Switch할 수 있다.")
print()
print("Switch 확률 = (N-1)/N")
print()

door_counts = [3, 5, 10, 50, 100, 1000]
print(f"{'문 개수 N':>10} {'Stay 확률':>15} {'Switch 확률':>15}")
print("-" * 45)
for n_doors in door_counts:
    p_stay = 1 / n_doors
    p_switch = (n_doors - 1) / n_doors
    print(f"{n_doors:>10} {p_stay:>15.4f} {p_switch:>15.4f}")

print()
print("100개 문 직관:")
print("  문 1개 선택 → 진행자가 98개 문을 열어 염소를 보여줌")
print("  → 남은 1개 문에 차가 있을 확률 = 99/100 = 99%!")
print("  → 당연히 Switch해야 한다!")
print("  → 3개 문일 때도 원리는 동일: Switch = 2/3")

In [ ]:
# === N개 문에서의 Switch 확률 시각화 ===

fig, ax = plt.subplots(figsize=(10, 5))

n_doors_range = np.arange(3, 101)
switch_probs = (n_doors_range - 1) / n_doors_range
stay_probs = 1 / n_doors_range

ax.plot(n_doors_range, switch_probs, color='#2980b9', linewidth=2.5,
        label='Switch 확률 = (N-1)/N')
ax.plot(n_doors_range, stay_probs, color='#e74c3c', linewidth=2.5,
        label='Stay 확률 = 1/N')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)

# N=3 포인트
ax.plot(3, 2/3, 'o', color='#2980b9', markersize=10, zorder=5)
ax.annotate('N=3\nSwitch=2/3', xy=(3, 2/3), xytext=(10, 0.55),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

# N=100 포인트
ax.plot(100, 99/100, 'o', color='#2980b9', markersize=10, zorder=5)
ax.annotate('N=100\nSwitch=99%', xy=(100, 0.99), xytext=(70, 0.85),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

# 영역 채우기
ax.fill_between(n_doors_range, switch_probs, stay_probs, alpha=0.1, color='#2980b9')

ax.set_xlabel('문의 개수 (N)', fontsize=12)
ax.set_ylabel('확률', fontsize=12)
ax.set_title('문 개수(N)에 따른 Stay vs Switch 확률', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 해석 - Part 1

**Monty Hall 문제의 핵심**:

1. 진행자는 **정보를 가진 상태**에서 문을 연다 (무작위가 아님!)
2. 진행자가 문을 여는 행위가 **확률을 재분배**한다
3. 처음 선택한 문의 확률(1/3)은 변하지 않고, 나머지 확률(2/3)이 남은 문에 집중
4. 따라서 **Switch가 항상 유리** (2/3 > 1/3)

| 전략 | 확률 | 근거 |
|------|------|------|
| Stay | 1/3 | 최초 선택이 맞을 확률 |
| Switch | 2/3 | 최초 선택이 틀릴 확률 = 나머지에 차가 있을 확률 |

---

## Part 2: Survivorship Bias (생존자 편향)

### WWII Abraham Wald 이야기

제2차 세계대전 중, 미군은 **귀환한 폭격기의 총탄 자국**을 분석하여 장갑을 강화하려 했습니다.

**군 지휘부의 직관**: 총탄 자국이 많은 부위(날개, 동체)를 강화하자!

**Abraham Wald의 통찰**: **아니다!** 총탄 자국이 **없는** 부위(엔진)를 강화해야 한다!

In [ ]:
# === Survivorship Bias: 항공기 다이어그램 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (ax, title, subtitle) in enumerate(zip(
    axes,
    ['군 지휘부의 관찰:\n귀환 항공기의 총탄 자국', 'Abraham Wald의 통찰:\n강화해야 할 부위'],
    ['(빨간 점 = 총탄 자국이 많은 곳)', '(파란 영역 = 총탄 자국이 없는 곳 = 강화 필요!)']
)):
    ax.set_xlim(-6, 6)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # 동체 (타원)
    fuselage = mpatches.Ellipse((0, 0), 8, 1.5, color='#d5d8dc', ec='black', lw=1.5)
    ax.add_patch(fuselage)
    
    # 날개
    wing_left = mpatches.FancyBboxPatch((-2.5, 0.5), 3, 1.8, 
                                        boxstyle="round,pad=0.1",
                                        color='#d5d8dc', ec='black', lw=1.5)
    wing_right = mpatches.FancyBboxPatch((-2.5, -2.3), 3, 1.8,
                                         boxstyle="round,pad=0.1",
                                         color='#d5d8dc', ec='black', lw=1.5)
    ax.add_patch(wing_left)
    ax.add_patch(wing_right)
    
    # 꼬리
    tail = mpatches.FancyBboxPatch((-4.5, -0.8), 1.2, 1.6,
                                   boxstyle="round,pad=0.1",
                                   color='#d5d8dc', ec='black', lw=1.5)
    ax.add_patch(tail)
    
    # 엔진 위치 (날개 아래)
    engine1 = mpatches.Circle((-0.5, 1.2), 0.4, color='#aab7b8', ec='black', lw=1.5)
    engine2 = mpatches.Circle((-0.5, -1.2), 0.4, color='#aab7b8', ec='black', lw=1.5)
    ax.add_patch(engine1)
    ax.add_patch(engine2)
    ax.text(-0.5, 1.2, 'E', ha='center', va='center', fontsize=8, fontweight='bold')
    ax.text(-0.5, -1.2, 'E', ha='center', va='center', fontsize=8, fontweight='bold')
    
    # 조종석
    cockpit = mpatches.Circle((3.5, 0), 0.5, color='#aed6f1', ec='black', lw=1.5)
    ax.add_patch(cockpit)
    ax.text(3.5, 0, 'C', ha='center', va='center', fontsize=8, fontweight='bold')
    
    if idx == 0:
        # 왼쪽: 총탄 자국 표시 (날개, 동체, 꼬리 - 엔진/조종석 제외)
        np.random.seed(42)
        # 동체 총탄
        for _ in range(20):
            bx = np.random.uniform(-3, 2.5)
            by = np.random.uniform(-0.5, 0.5)
            ax.plot(bx, by, 'o', color='#e74c3c', markersize=4, alpha=0.7)
        # 날개 총탄
        for _ in range(15):
            bx = np.random.uniform(-2, 0.5)
            by_sign = np.random.choice([-1, 1])
            by = by_sign * np.random.uniform(0.8, 2)
            ax.plot(bx, by, 'o', color='#e74c3c', markersize=4, alpha=0.7)
        # 꼬리 총탄
        for _ in range(8):
            bx = np.random.uniform(-4.5, -3.5)
            by = np.random.uniform(-0.5, 0.5)
            ax.plot(bx, by, 'o', color='#e74c3c', markersize=4, alpha=0.7)
    
    else:
        # 오른쪽: 강화 필요 부위 강조
        engine1_h = mpatches.Circle((-0.5, 1.2), 0.5, color='#3498db', alpha=0.5, ec='#2980b9', lw=3)
        engine2_h = mpatches.Circle((-0.5, -1.2), 0.5, color='#3498db', alpha=0.5, ec='#2980b9', lw=3)
        cockpit_h = mpatches.Circle((3.5, 0), 0.6, color='#3498db', alpha=0.5, ec='#2980b9', lw=3)
        ax.add_patch(engine1_h)
        ax.add_patch(engine2_h)
        ax.add_patch(cockpit_h)
        
        ax.annotate('엔진 피격\n→ 귀환 불가!', xy=(-0.5, 1.2), xytext=(2, 2.5),
                    fontsize=9, fontweight='bold', color='#2980b9',
                    arrowprops=dict(arrowstyle='->', color='#2980b9', lw=1.5))
        ax.annotate('조종석 피격\n→ 귀환 불가!', xy=(3.5, 0), xytext=(4.5, 2),
                    fontsize=9, fontweight='bold', color='#2980b9',
                    arrowprops=dict(arrowstyle='->', color='#2980b9', lw=1.5))
    
    ax.set_title(title + '\n' + subtitle, fontsize=11, fontweight='bold')

plt.suptitle('Survivorship Bias: Abraham Wald의 항공기 분석',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 조건부확률로 설명 ===

print("=" * 60)
print(" Survivorship Bias의 조건부확률 분석")
print("=" * 60)
print()
print("핵심 조건부확률:")
print()
print("  P(귀환 | 날개 피격) = 높음")
print("    → 날개에 맞아도 비행 가능 → 귀환 데이터에 나타남")
print()
print("  P(귀환 | 엔진 피격) = 매우 낮음")
print("    → 엔진에 맞으면 추락 → 귀환 데이터에 나타나지 않음!")
print()
print("  P(귀환 | 조종석 피격) = 매우 낮음")
print("    → 조종사 사망 → 귀환 데이터에 나타나지 않음!")
print()
print("오류의 본질:")
print("  관찰 대상 = 귀환한 항공기만 (생존자만)")
print("  추락한 항공기(비생존자)의 데이터는 수집 불가")
print("  → 관찰 데이터가 전체를 대표하지 않음!")
print()

# 수치 예시
print("--- 가상 수치 예시 ---")
print()
data_surv = pd.DataFrame({
    '피격 부위': ['날개', '동체', '꼬리', '엔진', '조종석'],
    '총 피격 횟수': [100, 80, 60, 50, 30],
    '귀환 횟수': [90, 70, 50, 5, 2],
})
data_surv['귀환율'] = data_surv['귀환 횟수'] / data_surv['총 피격 횟수']
data_surv['귀환율(%)'] = (data_surv['귀환율'] * 100).round(1)

print(data_surv.to_string(index=False))
print()
print("→ 엔진/조종석 피격 시 귀환율이 극히 낮음")
print("→ 귀환 항공기에서 이 부위에 총탄 자국이 적은 이유!")
print("→ 따라서 이 부위를 강화해야 함!")

In [ ]:
# === 귀환율 시각화 ===

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- 왼쪽: 귀환 항공기에서 관찰된 총탄 자국 수 ---
ax1 = axes[0]
observed_damage = data_surv['귀환 횟수'].values
parts = data_surv['피격 부위'].values
colors_obs = ['#f39c12', '#f39c12', '#f39c12', '#3498db', '#3498db']

bars1 = ax1.barh(parts, observed_damage, color=colors_obs, edgecolor='white', linewidth=2)
for bar, val in zip(bars1, observed_damage):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', fontsize=11, fontweight='bold')

ax1.set_xlabel('귀환 항공기에서 관찰된 피격 수', fontsize=11)
ax1.set_title('군 지휘부가 본 데이터:\n귀환 항공기 총탄 자국 분포', fontsize=12, fontweight='bold')
ax1.invert_yaxis()

legend_handles = [
    mpatches.Patch(color='#f39c12', label='지휘부: "여기를 강화"'),
    mpatches.Patch(color='#3498db', label='Wald: "여기를 강화!"')
]
ax1.legend(handles=legend_handles, fontsize=9)

# --- 오른쪽: 실제 귀환율 ---
ax2 = axes[1]
survival_rates = data_surv['귀환율'].values
colors_rate = ['#27ae60', '#27ae60', '#f39c12', '#e74c3c', '#e74c3c']

bars2 = ax2.barh(parts, survival_rates * 100, color=colors_rate, edgecolor='white', linewidth=2)
for bar, val in zip(bars2, survival_rates):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{val*100:.0f}%', va='center', fontsize=11, fontweight='bold')

ax2.set_xlabel('피격 시 귀환율 (%)', fontsize=11)
ax2.set_title('Wald의 분석:\n피격 부위별 귀환율', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.set_xlim(0, 110)

plt.tight_layout()
plt.show()

### 해석 - Part 2

**Survivorship Bias의 핵심 논리:**

$$P(\text{총탄 자국 관찰} \mid \text{부위}) = P(\text{귀환} \mid \text{해당 부위 피격})$$

총탄 자국이 **많은** 곳 = 피격되어도 귀환 가능한 곳 (강화 불필요)  
총탄 자국이 **없는** 곳 = 피격되면 귀환 불가능한 곳 (**강화 필수!**)

#### ML에서의 생존자 편향

| 상황 | 편향 |
|------|------|
| **학습 데이터 편향** | 성공 사례만 수집 → 실패 패턴 학습 불가 |
| **선택 편향** | 특정 그룹만 데이터 수집 → 일반화 실패 |
| **앱 리뷰 분석** | 리뷰 작성자만 분석 → 조용한 이탈 고객 무시 |
| **A/B 테스트** | 완료한 사용자만 분석 → 중도 이탈자 무시 |

---

## Part 3: 식중독 문제

### 문제 상황

> 식당에서 A, B, C 세 가지 음식을 제공했다.  
> 일부 손님이 식중독에 걸렸다.  
> **어떤 음식이 식중독의 원인인가?**

조건부확률을 이용한 역학 조사!

In [ ]:
# === Part 3: 식중독 문제 데이터 생성 ===

# 가상 데이터: 50명의 손님
np.random.seed(42)
n_people = 50

# 각 음식 섭취 여부 (1=섭취, 0=미섭취)
ate_A = np.random.binomial(1, 0.6, n_people)  # A: 60% 섭취
ate_B = np.random.binomial(1, 0.5, n_people)  # B: 50% 섭취
ate_C = np.random.binomial(1, 0.7, n_people)  # C: 70% 섭취

# 식중독 여부: 음식 B가 주 원인 (B를 먹으면 70% 확률로 식중독)
# A, C는 식중독과 약한 관련
prob_sick = 0.05 + 0.65 * ate_B + 0.05 * ate_A + 0.02 * ate_C
prob_sick = np.clip(prob_sick, 0, 1)
sick = np.array([np.random.binomial(1, p) for p in prob_sick])

# 데이터프레임 구성
df = pd.DataFrame({
    '음식A 섭취': ate_A,
    '음식B 섭취': ate_B,
    '음식C 섭취': ate_C,
    '식중독': sick
})

print(f"전체 손님: {n_people}명")
print(f"식중독 환자: {sick.sum()}명 ({sick.sum()/n_people*100:.0f}%)")
print()
print(df.head(10).to_string())
print("...")

In [ ]:
# === 2x2 Contingency Table + 조건부확률 ===

def analyze_food(df, food_col, sick_col='식중독'):
    """음식별 2x2 분할표와 조건부확률을 계산한다.
    
    Parameters
    ----------
    df : pd.DataFrame
        데이터
    food_col : str
        음식 섭취 컬럼명
    sick_col : str
        식중독 컬럼명
    
    Returns
    -------
    dict
        분석 결과 딕셔너리
    """
    # 2x2 분할표
    ct = pd.crosstab(df[food_col], df[sick_col], margins=True)
    ct.index = ['미섭취', '섭취', '합계']
    ct.columns = ['정상', '식중독', '합계']
    
    # 조건부확률
    ate = df[df[food_col] == 1]
    not_ate = df[df[food_col] == 0]
    
    p_sick_given_ate = ate[sick_col].mean() if len(ate) > 0 else 0
    p_sick_given_not_ate = not_ate[sick_col].mean() if len(not_ate) > 0 else 0
    
    # 상대위험도 (Relative Risk)
    rr = p_sick_given_ate / p_sick_given_not_ate if p_sick_given_not_ate > 0 else float('inf')
    
    return {
        'table': ct,
        'p_sick_ate': p_sick_given_ate,
        'p_sick_not_ate': p_sick_given_not_ate,
        'relative_risk': rr,
        'n_ate': len(ate),
        'n_not_ate': len(not_ate)
    }

# 각 음식별 분석
foods = ['음식A 섭취', '음식B 섭취', '음식C 섭취']
food_labels = ['음식 A', '음식 B', '음식 C']
results = {}

for food, label in zip(foods, food_labels):
    result = analyze_food(df, food)
    results[label] = result
    
    print(f"\n{'='*50}")
    print(f" {label} 분석")
    print(f"{'='*50}")
    print()
    print("[2x2 분할표]")
    print(result['table'])
    print()
    print(f"  P(식중독 | {label} 섭취)   = {result['p_sick_ate']:.3f} ({result['p_sick_ate']*100:.1f}%)")
    print(f"  P(식중독 | {label} 미섭취) = {result['p_sick_not_ate']:.3f} ({result['p_sick_not_ate']*100:.1f}%)")
    print(f"  상대위험도 (RR) = {result['relative_risk']:.2f}")
    print(f"    → RR > 1이면 해당 음식이 식중독 위험을 높임")

In [ ]:
# === 시각화: 조건부확률 비교 ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors_food = ['#3498db', '#e74c3c', '#27ae60']

for idx, (label, color) in enumerate(zip(food_labels, colors_food)):
    ax = axes[idx]
    r = results[label]
    
    categories = [f'{label}\n섭취', f'{label}\n미섭취']
    probs = [r['p_sick_ate'], r['p_sick_not_ate']]
    
    bars = ax.bar(categories, probs, color=[color, '#bdc3c7'],
                  edgecolor='white', linewidth=2, width=0.5)
    
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{prob:.1%}', ha='center', fontsize=13, fontweight='bold')
    
    ax.set_ylabel('P(식중독)', fontsize=11)
    ax.set_title(f'{label}\nRR = {r["relative_risk"]:.2f}', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('음식별 조건부확률: P(식중독 | 섭취) vs P(식중독 | 미섭취)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === 상대위험도 비교 차트 ===

fig, ax = plt.subplots(figsize=(8, 5))

rr_values = [results[label]['relative_risk'] for label in food_labels]
colors_rr = ['#3498db' if rr <= 1.5 else '#f39c12' if rr <= 3 else '#e74c3c' for rr in rr_values]

bars = ax.barh(food_labels, rr_values, color=colors_rr, edgecolor='white', linewidth=2, height=0.5)

for bar, rr in zip(bars, rr_values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'RR = {rr:.2f}', va='center', fontsize=12, fontweight='bold')

ax.axvline(x=1.0, color='black', linestyle='--', linewidth=2, label='RR = 1 (위험 없음)')
ax.set_xlabel('상대위험도 (Relative Risk)', fontsize=12)
ax.set_title('음식별 상대위험도 비교', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# 결론
max_rr_food = food_labels[np.argmax(rr_values)]
print(f"\n결론: {max_rr_food}의 상대위험도가 {max(rr_values):.2f}로 가장 높음")
print(f"→ {max_rr_food}가 식중독의 가장 유력한 원인!")

### 해석 - Part 3

**식중독 원인 분석의 핵심:**

| 지표 | 정의 | 판단 기준 |
|------|------|--------|
| **조건부확률** | $P(\text{식중독} \mid \text{음식 X 섭취})$ | 높을수록 의심 |
| **대조군** | $P(\text{식중독} \mid \text{음식 X 미섭취})$ | 비교 기준 |
| **상대위험도 (RR)** | $\frac{P(\text{식중독} \mid \text{섭취})}{P(\text{식중독} \mid \text{미섭취})}$ | RR > 1이면 위험 증가 |

**상대위험도 해석:**
- $RR = 1$: 섭취 여부와 식중독 무관
- $RR > 1$: 섭취 시 식중독 위험 증가
- $RR < 1$: 섭취 시 식중독 위험 감소 (보호 효과)

단순히 "식중독 환자 중 음식 X를 먹은 비율"만 보면 안 됩니다!  
**반드시** 먹지 않은 그룹(대조군)과 비교해야 합니다.

In [ ]:
# === 최종 요약 ===

print("=" * 65)
print(" A14. 조건부확률의 반직관적 결과 최종 요약")
print("=" * 65)
print()
print("[Part 1: Monty Hall 문제]")
print("  - Stay: 1/3, Switch: 2/3 → Switch가 2배 유리")
print("  - 진행자의 행동이 확률을 재분배")
print("  - 핵심: 조건부 정보가 확률을 바꾼다")
print()
print("[Part 2: Survivorship Bias]")
print("  - 귀환 항공기 = 생존자 (데이터의 일부)")
print("  - 총탄 자국이 없는 곳 = 맞으면 귀환 불가 = 강화 필요")
print("  - P(귀환|날개 피격) >> P(귀환|엔진 피격)")
print("  - ML 교훈: 데이터 편향을 항상 의심하라")
print()
print("[Part 3: 식중독 문제]")
print("  - P(식중독|X 섭취) vs P(식중독|X 미섭취) 비교")
print("  - 상대위험도(RR) > 1이면 해당 음식이 원인 후보")
print("  - 대조군과의 비교가 핵심 (단순 비율 X)")
print()
print("[공통 교훈]")
print("  1. 직관은 조건부확률에서 자주 실패한다")
print("  2. 데이터를 수집한 조건을 항상 고려하라")
print("  3. 비교 대상(대조군, 기저율)이 없는 분석은 불완전하다")
print("=" * 65)